In [1]:
'''
https://zhuanlan.zhihu.com/p/559824020  
'''

import torch

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'torch version: {torch.__version__}')

CUDA available: True
torch version: 2.5.0


In [2]:
'''
multiply == *
广播不仅可以“拷贝”一个标量，还可以“拷贝”一个向量、一个矩阵，或者张量
广播机制是从最后一个维度开始对齐，检查到不匹配的维度为1就会在这个维度上复制到匹配
'''
x = torch.randn(3, 1, 2, 5)
a = torch.randn(3, 6, 2, 1)
y = a * x
print(y.shape)

torch.Size([3, 6, 2, 5])


In [11]:
'''广播机制可以替代循环'''
a = torch.randn(3, 1)
b = torch.randn(1, 6)
print(a * b)
alist = a.squeeze().tolist()
blist = b.squeeze().tolist()
res = []
for i in alist:
    for j in blist:
        res.append(i * j)
ab = torch.tensor(res).reshape(len(alist), len(blist))
torch.equal(a * b, ab)

tensor([[ 0.5269, -0.0150, -1.0523, -0.3685,  0.4647,  0.0923],
        [-0.1930,  0.0055,  0.3854,  0.1350, -0.1702, -0.0338],
        [-0.4641,  0.0132,  0.9268,  0.3246, -0.4093, -0.0813]])


True

In [14]:
x = torch.randn(3, 4, 2, 5)
a = torch.randn(3, 4, 2, 1)
y = a * x
print(y.shape)

torch.Size([3, 4, 2, 5])


In [15]:
x = torch.randn(3, 4, 2, 5)
a = torch.randn(1, 4, 2, 1)
y = a * x
print(y.shape)

torch.Size([3, 4, 2, 5])


In [12]:
x = torch.randn(3, 4, 2, 5)
a = torch.randn(3, 4, 2, 5)
y = a * x
print(y.shape)

b = torch.randn(4, 2, 5)
y = x * b
print(y.shape)

c = torch.randn(2, 5)
y = x * c
print(y.shape)

d = torch.randn(5)
y = d * x
print(y.shape)

scaler = torch.randn(1)
y = x * scaler
print(y.shape)

torch.Size([3, 4, 2, 5])
torch.Size([3, 4, 2, 5])
torch.Size([3, 4, 2, 5])
torch.Size([3, 4, 2, 5])
torch.Size([3, 4, 2, 5])


# matmul 缩并一个维度
高维数组想做矩阵乘法时，需要保证前一个数组的最后一个维度`shape[-1]`和后一个数组的倒数第二个维度`shape[-2]`保持一致（如果后一个数组是向量就是倒数第一个维度）  
- 矩阵乘矩阵：`left.shape[-1]=right.shape[-2]`
- 向量乘矩阵：`left.shape[-1]=right.shape[-2]`
- 矩阵乘向量：`left.shape[-1]=right.shape[-1]`
- 向量乘向量：`left.shape[-1]=right.shape[-1]`

如果高维张量相乘，大于倒数两个维度的维度就是batch维度，不参与运算，只要求对齐，对齐也可以是广播机制对齐。  
$$
W^{abcd} \@ X^{df} = Y^{abcf}
$$  
### 存在问题
matmul的广播机制是采用copy data来expand到相同维度尺寸，因此存在broadcast的matmul性能还不如einsum，尽量自己手动broadcast一下。

In [23]:
# 向量乘矩阵
vec = torch.randn(3)
mat = torch.randn(3, 6)
print(vec.matmul(mat))

# 矩阵乘向量
print(mat.t().matmul(vec))
# 矩阵乘矩阵
row = vec.unsqueeze(0)
print(f'row matrix shape: {row.shape}')
print(row.matmul(mat))

col = vec.unsqueeze(1)
print(f'col matrix shape: {col.shape}')
print(mat.t().matmul(col))

# 向量乘向量
arr = torch.randn(3)
print(vec.matmul(arr))

tensor([ 2.6929,  2.0294, -1.8522,  0.9722, -2.7175, -0.4690])
tensor([ 2.6929,  2.0294, -1.8522,  0.9722, -2.7175, -0.4690])
row matrix shape: torch.Size([1, 3])
tensor([[ 2.6929,  2.0294, -1.8522,  0.9722, -2.7175, -0.4690]])
col matrix shape: torch.Size([3, 1])
tensor([[ 2.6929],
        [ 2.0294],
        [-1.8522],
        [ 0.9722],
        [-2.7175],
        [-0.4690]])
tensor(1.9234)


# tensordot 缩并/扩充多个维度


In [8]:
nums = [2, 7, 11, 15]
target = 9
diff = [target - i for i in nums]
print(diff)
idx = []
for i, v in enumerate(diff):
    if v in nums:
        idx.append(i)
print(idx)

[7, 2, -2, -6]
[0, 1]
